# API Testing Notebook

Manual smoke tests against a running `rescuevision-serve` container.

**Start the server first:**
```bash
make up          # docker-compose
# or
make run         # local uvicorn
```

In [ ]:
import json
import requests
import numpy as np
import cv2
from pathlib import Path

BASE = 'http://localhost:8080'

## Health & Readiness

In [ ]:
r = requests.get(f'{BASE}/health')
print(f'/health  → {r.status_code}')
print(json.dumps(r.json(), indent=2))

In [ ]:
r = requests.get(f'{BASE}/ready')
print(f'/ready  → {r.status_code}')
print(json.dumps(r.json(), indent=2))

## Model Info

In [ ]:
r = requests.get(f'{BASE}/model/info')
print(json.dumps(r.json(), indent=2))

## Predict — Synthetic Image

In [ ]:
img = np.zeros((480, 640, 3), dtype=np.uint8)
_, buf = cv2.imencode('.jpg', img)
img_bytes = buf.tobytes()

r = requests.post(
    f'{BASE}/predict',
    files={'file': ('test.jpg', img_bytes, 'image/jpeg')},
)
print(f'Status: {r.status_code}')
print(f'X-Request-ID: {r.headers.get("x-request-id")}')
body = r.json()
print(json.dumps(body, indent=2))

## Predict — Real Image from Val Set

In [ ]:
val_images = sorted(Path('data/coco_person/images/val').glob('*.jpg'))
if val_images:
    with open(val_images[0], 'rb') as f:
        r = requests.post(
            f'{BASE}/predict',
            files={'file': (val_images[0].name, f, 'image/jpeg')},
        )
    print(f'Status: {r.status_code}')
    body = r.json()
    print(f'Detections: {len(body["detections"])}')
    for d in body['detections']:
        print(f"  conf={d['confidence']:.3f}  bbox={d['bbox']}")
else:
    print('No val images found — run `make data` first')

## Error Cases

In [ ]:
# Wrong MIME type
r = requests.post(f'{BASE}/predict', files={'file': ('readme.txt', b'hello', 'text/plain')})
print(f'text/plain → {r.status_code}  detail: {r.json().get("detail")}')

# Corrupt image bytes
r = requests.post(f'{BASE}/predict', files={'file': ('bad.jpg', b'\xff\xd8garbage', 'image/jpeg')})
print(f'corrupt    → {r.status_code}  detail: {r.json().get("detail")}')

## Throughput Mini-Benchmark

In [ ]:
import time

N = 20
latencies = []
for _ in range(N):
    t0 = time.perf_counter()
    requests.post(f'{BASE}/predict', files={'file': ('t.jpg', img_bytes, 'image/jpeg')})
    latencies.append((time.perf_counter() - t0) * 1000)

print(f'N={N}  mean={np.mean(latencies):.1f} ms  p95={np.percentile(latencies, 95):.1f} ms  rps={1000/np.mean(latencies):.1f}')